---
title: "Capstone: Ship and Operate the Full-Stack Agent"
description: "Prove the browser-to-agent-to-database vertical slice, rehearse recovery, and turn dogfood evidence into the next release plan."
categories: [software-engineering, full-stack, agents, capstone, deployment, reliability]
---

The capstone ships the system promised by the course. Another learner should be able to install autocode, start one service, open the browser, create a session, stream a deterministic or live agent run, refresh into SQLite-backed history, and inspect the evidence when something fails. The release includes the application and the tests, migration notes, backup drill, and operating decisions required to trust it.


## Definition of done

The capstone release must demonstrate all of these paths:

- the built package contains Python modules and browser assets;
- `autocode serve` loads a responsive, keyboard-usable interface;
- REST creates, lists, and restores sessions through the application service;
- a WebSocket command produces persisted user, stream, assistant, and terminal events;
- two observers receive the same event ids and reconnect from a cursor;
- the journal repairs an interrupted SQLite projection without duplication;
- artifact, search, sync-auth, and background-job tests pass;
- the service distinguishes health, readiness, and dependency degradation;
- backup restore and support-bundle redaction drills succeed; and
- upgrade and rollback are exercised against copied representative data.

Each item needs an observed command, result, and artifact path. Unrun items remain unknown rather than passing by implication.


## Use a disaggregated release scorecard

A single “full stack works” result cannot locate failures. Track the frontend, transport, application, agent, data, concurrency, and operations boundaries separately, then add one end-to-end result that crosses them together.


In [1]:
scorecard = {
    "frontend_assets": "pass: root and static asset integration test",
    "rest_contract": "pass: create/list/detail integration test",
    "websocket_stream": "pass: durable command-to-terminal event test",
    "agent_boundary": "pass: deterministic runner; live mode requires credentials",
    "database_recovery": "pass: idempotent journal replay test",
    "multi_client": "pass: event-id convergence and cursor replay",
    "auth_and_sync": "pass: token rejection and idempotent push",
    "deployment": "unknown: run clean wheel and container rehearsal",
}

assert all(value.startswith(("pass:", "unknown:")) for value in scorecard.values())
print("\n".join(f"{boundary}: {result}" for boundary, result in scorecard.items()))


frontend_assets: pass: root and static asset integration test
rest_contract: pass: create/list/detail integration test
websocket_stream: pass: durable command-to-terminal event test
agent_boundary: pass: deterministic runner; live mode requires credentials
database_recovery: pass: idempotent journal replay test
multi_client: pass: event-id convergence and cursor replay
auth_and_sync: pass: token rejection and idempotent push
deployment: unknown: run clean wheel and container rehearsal


The explicit deployment unknown prevents source-checkout success from becoming a release claim. Add measured first-feedback latency, session-read percentiles, disk growth, queue depth, reconnect duration, and live-model cost only after running named workloads. Keep deterministic and live-agent evidence separate.


## Prove the vertical slice automatically

The capstone test starts from an empty temporary data directory and uses the same FastAPI application factory as Uvicorn. It loads the browser document, creates a session through REST, submits a command through WebSocket, receives the durable stream, and reloads the session through REST. No direct repository setup occurs after the server starts.


In [2]:
from tempfile import TemporaryDirectory

from fastapi.testclient import TestClient

from autocode.runner import DemoAgentRunner
from autocode_service.api import create_app

with TemporaryDirectory() as directory:
    app = create_app(
        database_path=f"{directory}/sessions.db",
        runner=DemoAgentRunner(),
        agent_mode="demo",
    )
    with TestClient(app) as client:
        assert client.get("/").status_code == 200
        created = client.post("/api/sessions", json={"title": "capstone"})
        session_id = created.json()["session_id"]

        with client.websocket_connect(f"/ws/sessions/{session_id}") as socket:
            socket.send_json({"type": "user_message", "content": "ship the vertical slice"})
            streamed = []
            while not streamed or streamed[-1]["kind"] != "run_finished":
                streamed.append(socket.receive_json())

        refreshed = client.get(f"/api/sessions/{session_id}").json()

assert created.status_code == 201
assert [event["event_id"] for event in refreshed["events"]] == [
    event["event_id"] for event in streamed
]
assert {"user_message", "text_delta", "assistant_message", "run_finished"} <= {
    event["kind"] for event in streamed
}
print("vertical slice persisted", len(streamed), "browser-visible events")


vertical slice persisted 11 browser-visible events


This test earns the course's full-stack claim because it crosses frontend delivery, REST, WebSocket, application orchestration, agent-runner translation, journal and SQLite persistence, broker publication, and refresh. It does not replace a real browser accessibility and responsive-layout pass; the two forms of evidence answer different questions.


## Dogfood, postmortem, and handoff

Run the release on a representative repository with the deterministic runner first, then the live harness when credentials and safety policy permit. Record frontend failures, protocol failures, application bugs, agent or tool failures, persistence failures, and operational failures separately. A timeline should name detection, user impact, durable evidence, mitigation, and owner.

The trilogy closes at a concrete boundary: model work makes the system capable, the harness makes model and tool behavior governable, and this full-stack product makes that behavior usable, durable, observable, and shippable.


## Exercises

Create a v1.1 decision memo from at least three observed capstone problems. Each row must cite evidence, identify the failed stack boundary, estimate user impact, choose a fix or deliberate non-fix, and define the browser, API, data, or recovery regression that will protect the decision.


### [P12.1] Write an evidence-backed v1.1 plan

For three dogfood or drill failures, classify the full-stack boundary, cite the exact evidence, choose a fix or deliberate non-fix, and define an automated or manual regression check with an owner.


In [3]:
#| echo: false
#| eval: false
#| output: false
# Sbe rnpu ceboyrz, erpbeq gur bofreirq irefvba naq pbasvthengvba, jbexybnq, hfre vzcnpg, snvyrq obhaqnel, rirag be ybt vqragvsvref, fperrafubg be pbzznaq bhgchg cngu, naq erpheerapr. Pynffvsl vg nf sebagraq cebwrpgvba, ERFG/JroFbpxrg genafcbeg, nccyvpngvba bepurfgengvba, ntrag/gbby orunivbe, crefvfgrapr, pbapheerapl, qrcyblzrag, be bcrengvbaf. Enax erpheevat uvtu-vzcnpg snvyherf svefg. Fgngr gur qrpvfvba, bjare, naq npprcgnapr qngr. N svk zhfg anzr gur aneebj erterffvba cngu, fhpu nf n oebjfre sbphf purpx, UGGC pbagenpg grfg, JroFbpxrg ercynl grfg, zvtengvba qevyy, be erfgber grfg. N qryvorengr aba-svk zhfg anzr gur fpbcr be pbfg obhaqnel naq gur rivqrapr guerfubyq gung jbhyq erbcra gur qrpvfvba. Qb abg cebzbgr na haeha purpx gb cnff.